# Unidad 5 · Colab 1 de 3
## Inspección de sitios web y extracción estática con BeautifulSoup

**Objetivos de este notebook**

- Entender la estructura del DOM y el HTML de una página.
- Escribir selectores CSS y expresiones XPath para ubicar elementos.
- Hacer requests HTTP y parsear HTML con `requests` + `BeautifulSoup`.
- Extraer datos estructurados (texto, atributos, tablas) de páginas estáticas.
- Manejar paginación y errores básicos de red.

> **Nivel:** intermedio. Se asume que ya sabés Python (loops, funciones, listas/diccionarios) y HTML/CSS básico.

**Sitios de práctica:** este notebook usa [quotes.toscrape.com](http://quotes.toscrape.com) y [books.toscrape.com](http://books.toscrape.com), sitios creados específicamente para practicar scraping sin problemas legales ni éticos.

---

## 1. Inspección de sitios web: DOM y HTML

El **DOM (Document Object Model)** es la representación en forma de árbol que el navegador construye a partir del HTML. Cada etiqueta es un *nodo*, con padres, hijos y hermanos. Entender esa jerarquía es la base para poder ubicar cualquier dato dentro de una página.

Para inspeccionar un sitio: click derecho → **Inspeccionar** (o `F12`) en Chrome/Firefox abre las DevTools, donde podés ver el HTML renderizado, el panel *Elements*, y usar la lupa para seleccionar un elemento visual y saltar directo a su nodo en el DOM.

Documentación oficial: [Introducción al DOM (MDN)](https://developer.mozilla.org/es/docs/Web/API/Document_Object_Model/Introduction) · [Cómo funciona el HTML (MDN)](https://developer.mozilla.org/es/docs/Web/HTML)

## 2. CSS Selectors

Los selectores CSS permiten ubicar elementos del DOM por etiqueta, clase, id, atributo o posición.

| Selector | Ejemplo | Selecciona |
|---|---|---|
| Etiqueta | `p` | Todos los `<p>` |
| Clase | `.precio` | Elementos con clase precio |
| ID | `#header` | El elemento con id header |
| Descendiente | `div.card h2` | `h2` dentro de `div.card` |
| Atributo | `a[href]` | `<a>` que tengan atributo href |
| Pseudo-clase | `li:first-child` | El primer `li` de su padre |

Documentación oficial: [Selectores CSS (MDN)](https://developer.mozilla.org/es/docs/Web/CSS/CSS_selectors)

### Ejercicio 1 — Selectores CSS

Dado este fragmento de HTML:

```html
<div class='producto'>
  <h2 class='titulo'>Notebook Lenovo</h2>
  <span class='precio' data-moneda='USD'>599</span>
  <a href='/producto/123'>Ver más</a>
</div>
```

Escribí el selector CSS para: (a) el `h2` con clase `titulo`, (b) el `span` con clase `precio`, (c) el enlace que tiene atributo `href`.

<details>
<summary>💡 Ver solución</summary>

- (a) `.titulo` o `h2.titulo`
- (b) `.precio` o `span.precio`
- (c) `a[href]`

</details>

## 3. XPath

XPath es un lenguaje más expresivo que CSS: permite navegar hacia arriba en el árbol (padres/ancestros), filtrar por texto y usar condiciones lógicas — cosas que CSS no puede hacer.

| XPath | Selecciona |
|---|---|
| `//h2` | Todos los `<h2>` del documento |
| `//div[@class='producto']` | `div` con clase exacta producto |
| `//span[@class='precio']/text()` | El texto dentro del `span.precio` |
| `//a[contains(@href, '/producto/')]` | Links cuyo href contiene esa cadena |
| `//div[h2[text()='Notebook Lenovo']]` | El `div` que contiene un `h2` con ese texto exacto |

Documentación oficial: [XPath (MDN)](https://developer.mozilla.org/es/docs/Web/XPath)

**¿CSS o XPath?** CSS es más legible para casos simples; XPath es necesario cuando necesitás filtrar por texto, subir a un ancestro, o aplicar condiciones complejas. Playwright y Selenium (Colab 2) soportan ambos.

### Ejercicio 2 — XPath

Usando el mismo HTML del Ejercicio 1, escribí una expresión XPath que seleccione el `div.producto` que contiene un `h2` cuyo texto sea exactamente Notebook Lenovo.

<details>
<summary>💡 Ver solución</summary>

```text
//div[@class='producto'][h2[text()='Notebook Lenovo']]
```

</details>

## 4. Extracción con `requests` y `BeautifulSoup`

Para páginas **estáticas** (el HTML que llega en la respuesta ya tiene todo el contenido, sin necesitar JavaScript), alcanza con pedir el HTML por HTTP y parsearlo.

```python
import requests
from bs4 import BeautifulSoup

resp = requests.get('http://quotes.toscrape.com')
resp.raise_for_status()
soup = BeautifulSoup(resp.text, 'html.parser')
print(soup.title.text)
```

Documentación oficial: [requests](https://requests.readthedocs.io/en/latest/) · [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)

In [ ]:
!pip install -q beautifulsoup4 requests lxml

import requests
from bs4 import BeautifulSoup

resp = requests.get('http://quotes.toscrape.com')
resp.raise_for_status()
soup = BeautifulSoup(resp.text, 'html.parser')

# TODO Ejercicio 3: extrae, para cada bloque .quote, el texto de la cita (.text)
# y el autor (.author), y guardalos en una lista de diccionarios
# [{'cita': ..., 'autor': ...}, ...]
quotes = soup.select('.quote')
print(f'Encontre {len(quotes)} citas en la pagina')

In [ ]:
# Solucion -- Ejercicio 3
datos = []
for q in soup.select('.quote'):
    texto = q.select_one('.text').get_text(strip=True)
    autor = q.select_one('.author').get_text(strip=True)
    datos.append({'cita': texto, 'autor': autor})

datos[:3]

## 5. Paginación y headers

Muchos sitios muestran resultados en varias páginas (`?page=2`, `/page/2/`, etc.). También conviene enviar un `User-Agent` propio, ya que algunos servidores bloquean el user-agent por defecto de `requests`.

```python
headers = {'User-Agent': 'Mozilla/5.0 (compatible; CursoScrapingBot/1.0)'}
resp = requests.get(url, headers=headers, timeout=10)
```

### Ejercicio 4 — Recorrer todas las páginas de quotes.toscrape.com

El sitio tiene paginación en `http://quotes.toscrape.com/page/2/`, `/page/3/`, etc., hasta que ya no hay botón *Next*. Completá el código para recorrer todas las páginas y juntar todas las citas.

In [ ]:
import time

todas = []
pagina = 1
headers = {'User-Agent': 'Mozilla/5.0 (compatible; CursoScrapingBot/1.0)'}

# TODO: recorre paginas hasta que soup.select_one('li.next') sea None
# Tip: usa time.sleep(1) entre requests (buena practica, se profundiza en el Colab 3)

<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import time

todas = []
pagina = 1
headers = {'User-Agent': 'Mozilla/5.0 (compatible; CursoScrapingBot/1.0)'}

while True:
    url = f'http://quotes.toscrape.com/page/{pagina}/'
    resp = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(resp.text, 'html.parser')
    for q in soup.select('.quote'):
        todas.append({
            'cita': q.select_one('.text').get_text(strip=True),
            'autor': q.select_one('.author').get_text(strip=True),
        })
    if soup.select_one('li.next') is None:
        break
    pagina += 1
    time.sleep(1)

len(todas)
```

</details>

## 6. Manejo de errores comunes

```python
try:
    resp = requests.get(url, headers=headers, timeout=10)
    resp.raise_for_status()   # lanza una excepcion si el status es 4xx/5xx
except requests.exceptions.Timeout:
    print('El servidor tardo demasiado en responder')
except requests.exceptions.HTTPError as e:
    print(f'Error HTTP: {e}')
```

Un selector que no encuentra nada devuelve `None` (con `select_one`) o una lista vacía (con `select`) — siempre conviene chequear antes de acceder a `.text` para evitar `AttributeError`.

## Mini-proyecto: catálogo de libros

Usando [books.toscrape.com](http://books.toscrape.com):

1. Extraé de la página principal: título, precio y disponibilidad de cada libro.
2. Recorré las categorías (o la paginación) para juntar al menos 100 libros.
3. Guardá el resultado en una lista de diccionarios y exportalo a un `.csv` con `pandas` o el módulo `csv`.

**Entregable:** un archivo `libros.csv` con al menos las columnas `titulo`, `precio`, `disponibilidad`.

---

**Seguís en:** *Colab 2 — Automatización de navegación con Playwright y Selenium*